# 06 — Registry: publicação e aprovação por IaC

Referência: [AgentCore workshops](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops). O código e o IaC desta jornada adaptam os conceitos às contas do workshop.

## Pré-requisito: 1 conta AWS

Esta aula não exige fronteiras entre contas. Seus stacks, integrações e permissões precisam estar implantados na conta da plataforma. Ela não funciona automaticamente em uma conta AWS vazia.

**Runner CodeBuild do projeto:** a seleção é declarada no deploy. A jornada `single-account` inclui 00–09 e 15–16; Registry, Terraform e RAG são exigidos somente pelas aulas que os utilizam. Ampliar a seleção exige atualizar a implantação pelo IaC. Confira a [matriz de contas e execução](../docs/contas-e-elegibilidade.md).

![Arquitetura AWS da aula 06](../docs/diagramas/svg/06_registry_governanca.svg)

Leia as fronteiras do diagrama e confronte-as com o IaC e as chamadas abaixo.

Todos os serviços e agentes desta jornada são implantados na AWS.
O notebook é o roteiro e cliente das APIs; ferramentas, inferência,
políticas, fontes e memória executam nos serviços indicados.
Use o ambiente AWS descrito no [guia de implantação](../docs/deploy.md).
Configure os identificadores dos stacks antes de executar. Uma
dependência ausente bloqueia a aula antes das chamadas do exercício.
Antes de usar “Run All”, selecione as aulas compatíveis com as contas
preparadas. O bloco 10–14 exige quatro contas na implantação atual.

### Carregar a configuração e conferir as dependências

A configuração contém perfis/roles e IDs, nunca senhas ou tokens. O STS confirma a conta antes de consultar os stacks; o manifesto exige as dependências desta aula.

In [ ]:
import json
import os
import uuid
from pathlib import Path
from IPython.display import Code, display
import inspect
from santander_demo.aws_workshop.client import AWSWorkshop
from santander_demo.aws_workshop.config import load
from santander_demo.aws_workshop.manifest import require

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CONFIG = Path(os.environ.get("WORKSHOP_CONFIG", ROOT / "config/aws.workshop.json"))
config = load(CONFIG)
dependencies = require(config, "06")
print({"lab": "06", "dependencies": list(dependencies), "region": config["region"]})

### Carregar o catálogo implantado

O catálogo tem roles diferentes para desenvolver, publicar, aprovar e consumir. Os SDKs atuais são agent-registry-control para governança e agent-registry para descoberta.

In [ ]:
from santander_demo.registry_workflow.client import RegistryLab
registry_config = json.loads((ROOT / dependencies["registry"]["config"]).read_text())
registry = RegistryLab(registry_config)
assert registry_config["RegistryArn"].startswith("arn:aws:agent-registry:")
print({"registry": registry_config["RegistryArn"], "states": registry.states()})

### Ler o ciclo declarativo de publicação

publish cria o pacote, submit o coloca em revisão e approve aplica uma decisão com o digest revisado. São stacks separados, executados pelo pipeline scripts/registry_deploy.py. Inspecione o recurso e a role responsáveis por cada transição.

In [ ]:
for phase in ["publish", "submit", "approve"]:
    document = json.loads((ROOT / "infra/registry_workflow/templates" / f"{phase}.json").read_text())
    print(phase)
    display(Code(json.dumps(document, indent=2), language="json"))

### Verificar estado e histórico do CloudFormation

Consultamos os registros atuais e os stacks responsáveis pelas transições. Reabrir esta aula não publica nem aprova outro pacote: para uma nova versão, altere o bundle e execute as fases pelo pipeline, revisando o digest antes de approve.

In [ ]:
states = registry.states()
assert set(states.values()) == {"APPROVED"}
cfn = registry.operator.client("cloudformation")
history = {}
for phase in ["publish", "submit", "approve"]:
    stack = cfn.describe_stacks(StackName=registry_config["prefix"] + "-" + phase)["Stacks"][0]
    assert stack["StackStatus"] in {"CREATE_COMPLETE", "UPDATE_COMPLETE"}
    history[phase] = {"stack": stack["StackName"], "status": stack["StackStatus"]}
print({"records": states, "pipeline": history})

**Separação de responsabilidades:** a aprovação autoriza publicar metadados no catálogo. IAM e Policy continuam decidindo se uma capacidade pode ser invocada. Nenhuma sugestão de um modelo é aprovada automaticamente.

### Evidências e encerramento

Registre IDs de requisição e decisões, sem tokens ou senhas. Os recursos são compartilhados entre aulas: a remoção pertence ao pipeline de limpeza do projeto e deve respeitar essa dependência.